# 원하는 포즈로 이미지 만드는 도구참조 사진에서 **자세(골격)만** 뽑아내고, 거기에 원하는 프롬프트를 입혀 새 이미지를 만듭니다.참조 사진의 인물·옷·배경은 버리고 자세만 가져오는 것이 목표입니다.| 단계 | 하는 일 | 쓰는 것 ||---|---|---|| 1 | 참조 사진 → 골격 이미지 | OpenPose (`controlnet_aux`) || 2 | 골격 + 프롬프트 → 결과 이미지 | Stable Diffusion 1.5 + ControlNet |**실행 준비:** 상단 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한 뒤,위에서부터 순서대로 실행하세요. API 키는 필요하지 않습니다.**준비물:** 사람 전신이 나온 사진 2장 (자세가 서로 다른 것). 이 저장소의`samples/images1.jpg`, `samples/images2.jpg`를 내려받아 써도 됩니다.

---## 결과 미리보기<img src="samples/preview.png" width="900">여섯 칸을 한 장으로 합친 대조표입니다. 개별 파일로도 볼 수 있습니다.| 참조 사진 | 추출된 골격 | 생성 결과 ||:---:|:---:|:---:|| <img src="samples/images1.jpg" width="190"> | <img src="samples/pose_01.png" width="190"> | <img src="samples/output_01.png" width="190"> || 포즈 1 · 정면 직립 | 4.29% · 검출 성공 | 자리채움 || <img src="samples/images2.jpg" width="190"> | <img src="samples/pose_02.png" width="190"> | <img src="samples/output_02.png" width="190"> || 포즈 2 · 다리 꼰 자세 | 4.42% · 검출 성공 | 자리채움 |오른쪽 열 두 장은 아직 생성 이미지가 아닙니다. 아래 4~7단계를 실행하면같은 파일명으로 실제 결과가 저장되고, 이 표도 자동으로 새 이미지를 가리킵니다.

---## 0단계. 런타임 확인GPU가 안 잡히면 `cuda: False`가 나옵니다. 그 상태로도 돌아가지만 이미지 한 장에10분 이상 걸리니, 꼭 T4 GPU로 바꾼 뒤 진행하세요.

In [ ]:
# 이 셀이 하는 일: GPU가 런타임에 붙었는지 확인한다.!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음"import torchprint("torch:", torch.__version__)print("cuda :", torch.cuda.is_available())

---## 1단계. 설치Colab 기본 환경에는 `diffusers`와 `controlnet_aux`가 없습니다. 2~3분 걸립니다.설치 중 아래 두 메시지는 **무시해도 됩니다.**- `ERROR: pip's dependency resolver ...` / `timm 0.6.7` 다운그레이드 알림 —  `controlnet_aux`가 `timm<=0.6.7`을 요구해서 생기는 것으로, 이 노트북이 쓰는  OpenPose 기능에는 영향이 없습니다.- `The module 'mediapipe' is not installed` —  `controlnet_aux`의 다른 검출기(얼굴 메시)용 경고이며, OpenPose는 쓰지 않습니다.설치 후 런타임 재시작 팝업이 떠도 **재시작하지 않아도** 됩니다.

In [ ]:
# 이 셀이 하는 일: 포즈 추출(controlnet_aux)과 이미지 생성(diffusers)에 필요한 패키지를 설치한다.%pip install -q "diffusers>=0.31" "transformers>=4.44" "accelerate>=0.34" \                "controlnet_aux==0.0.9" "safetensors"print("설치 완료")

---## 2단계. 참조 사진 준비자세를 가져올 사진을 올립니다. 결과가 잘 나오는 사진의 조건은 이렇습니다.- 사람 **전신**이 나오고, 팔다리가 화면 밖으로 잘리지 않았다- 인물이 화면에서 충분히 크다 (작으면 골격이 안 잡힘)- 사람이 한 명이다 (여러 명이면 그만큼 다 생성됨)

In [ ]:
# 이 셀이 하는 일: 이미지를 불러오고 생성에 맞는 크기로 맞추는 헬퍼 함수를 정의한다.from PIL import Imageimport requests, io, osTARGET = 512  # 생성 해상도. 768로 올리면 품질은 좋아지지만 VRAM을 더 씁니다.def fit(img, target=TARGET):    """긴 변을 target에 맞추고, 가로·세로를 8의 배수로 정리한다 (SD 요구사항)."""    img = img.convert("RGB")    w, h = img.size    s = target / max(w, h)    w, h = int(w * s), int(h * s)    w, h = w - w % 8, h - h % 8    return img.resize((w, h), Image.LANCZOS)def load_from_upload():    """파일 선택창을 띄워 업로드한 이미지를 반환한다."""    from google.colab import files    up = files.upload()    name = next(iter(up))    print("업로드:", name)    return fit(Image.open(io.BytesIO(up[name])))def load_from_url(url):    """이미지 URL에서 직접 불러온다."""    r = requests.get(url, timeout=30, headers={"User-Agent": "pose-image-tool"})    r.raise_for_status()    return fit(Image.open(io.BytesIO(r.content)))print("헬퍼 준비 완료")

In [ ]:
# 이 셀이 하는 일: 첫 번째 참조 사진을 올려 pose_image_1에 담는다.# 실행하면 파일 선택창이 뜹니다. samples/images1.jpg를 고르면 됩니다.pose_image_1 = load_from_upload()# URL로 넣으려면 위 줄을 주석 처리하고 아래 줄을 쓰세요.# pose_image_1 = load_from_url("https://example.com/my_pose.jpg")print("크기:", pose_image_1.size)pose_image_1

---## 3단계. 포즈 추출OpenPose로 관절을 뽑습니다. 검은 배경에 알록달록한 막대기가 나오면 성공입니다.이 골격 이미지가 다음 단계에서 ControlNet의 조건으로 들어갑니다.- `include_hand` / `include_face`를 켜면 손가락·얼굴 방향까지 잡습니다. 대신 느립니다.- 골격이 **텅 빈 검은 이미지**로 나오면 인물이 너무 작거나 잘린 경우입니다. 다른 사진을 쓰세요.- 손 그리기에는 `matplotlib`이 필요한데 Colab에 기본 설치되어 있습니다.

In [ ]:
# 이 셀이 하는 일: OpenPose 검출기를 내려받고, 사진에서 골격을 뽑는 함수를 정의한다.from controlnet_aux import OpenposeDetectoropenpose = OpenposeDetector.from_pretrained("lllyasviel/Annotators")def extract_pose(img):    """사진에서 골격 이미지를 뽑아 원본과 같은 크기로 돌려준다."""    pose = openpose(        img,        include_body=True,        include_hand=True,        include_face=True,        detect_resolution=512,        image_resolution=max(img.size),    )    return pose.resize(img.size, Image.LANCZOS)print("OpenPose 준비 완료")

In [ ]:
# 이 셀이 하는 일: 첫 번째 사진의 골격을 뽑고, 검출이 됐는지 숫자로 확인한다.import numpy as nppose_map_1 = extract_pose(pose_image_1)# 검은 배경이 아닌 픽셀의 비율. 1% 미만이면 검출에 실패한 것으로 본다.ratio = float((np.asarray(pose_map_1).max(axis=2) > 24).mean())print(f"골격 픽셀 비율: {ratio:.2%}  ->  {'검출 성공' if ratio > 0.01 else '검출 실패: 다른 사진을 쓰세요'}")pose_map_1

In [ ]:
# 이 셀이 하는 일: 원본 사진과 추출된 골격을 나란히 놓고 눈으로 비교한다.import matplotlib.pyplot as pltfig, ax = plt.subplots(1, 2, figsize=(9, 5))for a, img, t in zip(ax, [pose_image_1, pose_map_1], ["reference photo", "openpose skeleton"]):    a.imshow(img); a.set_title(t); a.axis("off")plt.tight_layout(); plt.show()

---## 4단계. 생성 파이프라인 로드ControlNet(포즈 조건)과 Stable Diffusion 1.5(그림 생성)를 묶어 올립니다.처음 실행 시 모델 약 4GB를 내려받습니다 (3~5분). 두 번째부터는 캐시를 씁니다.

In [ ]:
# 이 셀이 하는 일: ControlNet(OpenPose) + SD1.5 파이프라인을 GPU에 올린다.# torch는 0단계에서도 import 하지만, 그 셀을 건너뛰고 와도 동작하도록 여기서 다시 가져온다.import torchfrom diffusers import (    ControlNetModel,    StableDiffusionControlNetPipeline,    UniPCMultistepScheduler,)DEVICE = "cuda" if torch.cuda.is_available() else "cpu"DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32controlnet = ControlNetModel.from_pretrained(    "lllyasviel/control_v11p_sd15_openpose", torch_dtype=DTYPE)pipe = StableDiffusionControlNetPipeline.from_pretrained(    # runwayml/stable-diffusion-v1-5는 이전되어 지금은 리다이렉트로만 접근됩니다.    # 아래가 현재 정식 이름입니다.    "stable-diffusion-v1-5/stable-diffusion-v1-5",    controlnet=controlnet,    torch_dtype=DTYPE,    safety_checker=None,            # Colab 메모리 절약. 필요하면 이 줄을 지우세요.    requires_safety_checker=False,)pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)if DEVICE == "cuda":    pipe.enable_attention_slicing()   # VRAM 부족(OOM) 방지    # 주의: enable_model_cpu_offload()가 모델을 알아서 GPU로 옮깁니다.    #      여기서 pipe.to("cuda")를 추가로 부르면 diffusers가 오류를 냅니다.    pipe.enable_model_cpu_offload()else:    pipe = pipe.to(DEVICE)print("파이프라인 준비 완료 ->", DEVICE, DTYPE)

---## 5단계. 한 장 만들어 보기파라미터는 네 개만 기억하면 됩니다.| 파라미터 | 뜻 | 추천 ||---|---|---|| `steps` | 디노이징 횟수 | 25 (손이 깨지면 35) || `guidance` | 프롬프트 충실도 | 7.5 || `pose_scale` | 자세 유지 강도 | 1.0 (0.5~1.5) || `seed` | 난수 고정값 | 조건만 바꿔 비교할 때는 고정 |

In [ ]:
# 이 셀이 하는 일: 골격과 프롬프트를 받아 이미지 한 장을 만드는 함수를 정의한다.NEGATIVE = (    "lowres, bad anatomy, bad hands, extra fingers, fewer fingers, missing limbs, "    "extra limbs, deformed, disfigured, mutated, cropped, worst quality, low quality, "    "jpeg artifacts, blurry, watermark, signature, text")def generate(prompt, pose, negative=NEGATIVE, steps=25,             guidance=7.5, pose_scale=1.0, seed=1234):    """골격 이미지 + 프롬프트 -> 결과 이미지 1장."""    gen = torch.Generator(device="cpu").manual_seed(seed)    out = pipe(        prompt=prompt,        negative_prompt=negative,        image=pose,        num_inference_steps=steps,        guidance_scale=guidance,        controlnet_conditioning_scale=float(pose_scale),        generator=gen,    )    return out.images[0]print("generate() 준비 완료")

In [ ]:
# 이 셀이 하는 일: 첫 번째 골격으로 이미지 한 장을 실제로 만들어 본다.BASE_PROMPT = (    "a young man in a white linen shirt standing in a sunflower field, "    "golden hour, soft rim light, 85mm photo, shallow depth of field, "    "highly detailed, photorealistic")result = generate(BASE_PROMPT, pose_map_1, seed=1234)result

---## 6단계. 조건을 바꿔 보기 (실험 1) — 같은 포즈, 프롬프트만 바꾸기골격은 그대로 두고 프롬프트만 갈아 끼웁니다. 인물과 배경은 완전히 달라지는데자세는 유지되는지 확인합니다. T4에서 512px / 25 steps 기준 한 장당 약 15초.

In [ ]:
# 이 셀이 하는 일: 같은 골격에 서로 다른 프롬프트 4개를 넣어 결과를 비교한다.PROMPTS = {    "photo": BASE_PROMPT,    "knight": (        "a medieval knight in polished steel armor, standing in a castle courtyard, "        "dramatic overcast light, cinematic, highly detailed"    ),    "astronaut": (        "an astronaut in a white spacesuit standing on mars, dusty red terrain, "        "lens flare, cinematic lighting, highly detailed"    ),    "anime": (        "anime illustration of a young man standing, clean lineart, cel shading, "        "pastel colors, studio ghibli inspired"    ),}results = {}for name, p in PROMPTS.items():    print(f"생성 중: {name}")    results[name] = generate(p, pose_map_1, seed=1234)fig, ax = plt.subplots(1, len(results) + 1, figsize=(3.2 * (len(results) + 1), 4))ax[0].imshow(pose_map_1); ax[0].set_title("pose"); ax[0].axis("off")for a, (name, img) in zip(ax[1:], results.items()):    a.imshow(img); a.set_title(name); a.axis("off")plt.tight_layout(); plt.show()

### 실험 2 — 자세 유지 강도(`pose_scale`) 바꾸기프롬프트를 고정하고 `pose_scale`만 바꿉니다. 이 값이 "프롬프트와 골격 중어느 쪽을 더 따를지"를 정합니다.

In [ ]:
# 이 셀이 하는 일: 같은 프롬프트에 pose_scale만 0.5/1.0/1.5로 바꿔 자세 추종 정도를 비교한다.DANCER = ("a dancer mid-motion on a dark stage, single spotlight, dust in the air, "          "dramatic shadows, photorealistic, highly detailed")scales = [0.5, 1.0, 1.5]comp = [generate(DANCER, pose_map_1, pose_scale=s, seed=7) for s in scales]fig, ax = plt.subplots(1, len(scales) + 1, figsize=(3.2 * (len(scales) + 1), 4))ax[0].imshow(pose_map_1); ax[0].set_title("pose"); ax[0].axis("off")for a, img, s in zip(ax[1:], comp, scales):    a.imshow(img); a.set_title(f"pose_scale={s}"); a.axis("off")plt.tight_layout(); plt.show()

### 실험 3 — 같은 프롬프트, 포즈 사진만 바꾸기이번엔 반대로 프롬프트를 고정하고 참조 사진을 바꿉니다. 두 번째 사진을 올려골격을 뽑고, 실험 1의 기준 프롬프트를 그대로 적용합니다.

In [ ]:
# 이 셀이 하는 일: 두 번째 참조 사진을 올려 골격까지 한 번에 뽑는다.# samples/images2.jpg처럼 첫 번째와 자세가 다른 사진을 고르세요.pose_image_2 = load_from_upload()pose_map_2 = extract_pose(pose_image_2)ratio2 = float((np.asarray(pose_map_2).max(axis=2) > 24).mean())print(f"골격 픽셀 비율: {ratio2:.2%}  ->  {'검출 성공' if ratio2 > 0.01 else '검출 실패'}")fig, ax = plt.subplots(1, 2, figsize=(9, 5))for a, img, t in zip(ax, [pose_image_2, pose_map_2], ["reference photo 2", "openpose skeleton 2"]):    a.imshow(img); a.set_title(t); a.axis("off")plt.tight_layout(); plt.show()

In [ ]:
# 이 셀이 하는 일: 동일한 프롬프트를 두 골격에 각각 넣어, 포즈만 바뀌었을 때의 차이를 본다.result_pose2 = generate(BASE_PROMPT, pose_map_2, seed=1234)panels = [(pose_map_1, "pose 1"), (results["photo"], "output 1"),          (pose_map_2, "pose 2"), (result_pose2, "output 2")]fig, ax = plt.subplots(1, 4, figsize=(13, 4))for a, (img, t) in zip(ax, panels):    a.imshow(img); a.set_title(t); a.axis("off")plt.suptitle("same prompt, different pose")   # 한글은 matplotlib 기본 폰트에 없어 □로 깨집니다plt.tight_layout(); plt.show()

---## 6-4단계. 내 프롬프트로 직접 해보기 (작성 칸)여기가 직접 채우는 칸입니다. 아래 `MY_PROMPTS`의 문장을 바꿔 실행하면그 자리에서 결과가 나옵니다. 몇 개를 적어도 되고, 한 개만 남겨도 됩니다.- 자세는 ControlNet이 잡아주니 자세 설명을 길게 쓸 필요는 없습니다.- `MY_POSE`로 어느 골격을 쓸지 고릅니다 (`pose_map_1` 또는 `pose_map_2`).- 이름(키)은 **영문으로** 적으세요. 그래프 제목에 그대로 쓰이는데, matplotlib  기본 폰트에 한글이 없어 `시도1`은 `□□1`로 깨집니다.- 마음에 드는 조합이 나오면 `prompts.md`의 "작성 칸"에 옮겨 적어 두세요.- 프롬프트 조각 예시는 `prompts.md` 맨 아래에 모아 두었습니다.

In [ ]:
# 이 셀이 하는 일: 내가 쓴 프롬프트로 이미지를 만들어 한 줄로 늘어놓고 보여준다.# 아래 세 곳만 고치면 됩니다: MY_PROMPTS / MY_POSE / MY_PARAMSMY_PROMPTS = {    # "이름": "프롬프트를 여기에"    # 이름은 그래프 제목으로 쓰이니 영문으로 적으세요.    # matplotlib 기본 폰트에 한글이 없어 "시도1" 같은 이름은 □□1 로 깨집니다.    "try1": "a young man in a heavy winter coat standing on a snowy street, "            "overcast light, 50mm photo, highly detailed, photorealistic",    "try2": "oil painting of a young man standing, chiaroscuro, "            "visible brush strokes, museum quality",}MY_POSE = pose_map_1          # pose_map_1 또는 pose_map_2MY_PARAMS = dict(steps=25, guidance=7.5, pose_scale=1.0, seed=1234)my_results = {}for name, p in MY_PROMPTS.items():    print(f"생성 중: {name}")    my_results[name] = generate(p, MY_POSE, **MY_PARAMS)fig, ax = plt.subplots(1, len(my_results) + 1, figsize=(3.2 * (len(my_results) + 1), 4))ax[0].imshow(MY_POSE); ax[0].set_title("pose"); ax[0].axis("off")for a, (name, img) in zip(ax[1:], my_results.items()):    a.imshow(img); a.set_title(name); a.axis("off")plt.tight_layout(); plt.show()

### 프롬프트 작성 칸아래 표를 **이 셀에서 직접 고쳐 쓰세요.** 셀을 두 번 누르면 편집 상태가 되고,`Esc`를 누르면 다시 표로 보입니다. 위 코드 셀의 `MY_PROMPTS`에 옮겨 담아 실행하고,나온 결과를 오른쪽 칸에 한 줄로 적어 두면 기록이 쌓입니다.| # | 프롬프트 | 포즈 | pose_scale | seed | 결과 ||---|---|---|---|---|---|| 1 | | pose_map_1 | 1.0 | 1234 | || 2 | | | | | || 3 | | | | | || 4 | | | | | || 5 | | | | | |**프롬프트 조각** — 필요한 것만 골라 이어 붙이면 됩니다.| 항목 | 쓸 수 있는 표현 ||---|---|| 인물 | `a young man` · `a young woman` · `an elderly fisherman` · `a knight` · `an astronaut` || 옷 | `in a white linen shirt` · `in polished steel armor` · `in a heavy winter coat` · `in a business suit` || 장소 | `in a sunflower field` · `in a castle courtyard` · `on a rainy neon street` · `in an empty gallery` || 빛 | `golden hour` · `soft rim light` · `dramatic overcast light` · `single spotlight` || 화풍 | `85mm photo, shallow depth of field` · `oil painting, visible brush strokes` · `anime illustration, cel shading` || 품질 | `highly detailed` · `photorealistic` · `cinematic` · `museum quality` |더 긴 기록은 [`prompts.md`](prompts.md)의 "다음에 시도할 프롬프트 (작성 칸)"에 남기세요.

In [ ]:
# 이 셀이 하는 일: 위에서 마음에 든 결과 한 장을 파일로 따로 저장한다.# my_results의 키 이름을 적으세요.PICK = "try1"os.makedirs("samples", exist_ok=True)out_name = f"samples/my_{PICK}.png"my_results[PICK].save(out_name)print("저장:", out_name, os.path.getsize(out_name) // 1024, "KB")

---## 7단계. 결과 저장제출용 파일 규칙에 맞춰 저장합니다.- `pose_01.png`, `pose_02.png` — ControlNet에 들어간 **골격 이미지**- `output_01.png`, `output_02.png` — 그 골격으로 만든 **결과 이미지**마지막 셀을 실행하면 `samples.zip`이 내려받아집니다. 압축을 풀어 저장소의`samples/` 폴더에 덮어쓰면 됩니다.

In [ ]:
# 이 셀이 하는 일: 골격 2장과 결과 2장을 samples/ 파일명 규칙대로 저장한다.os.makedirs("samples", exist_ok=True)pose_map_1.save("samples/pose_01.png")results["photo"].save("samples/output_01.png")pose_map_2.save("samples/pose_02.png")result_pose2.save("samples/output_02.png")# 참조 사진 원본도 같이 남겨 둔다 (골격과 대조용).pose_image_1.save("samples/images1.jpg", quality=92)pose_image_2.save("samples/images2.jpg", quality=92)for f in sorted(os.listdir("samples")):    print(f, os.path.getsize(f"samples/{f}") // 1024, "KB")

In [ ]:
# 이 셀이 하는 일: samples 폴더를 압축해 내 컴퓨터로 내려받는다.!zip -qr samples.zip samplesfrom google.colab import filesfiles.download("samples.zip")

---## 8단계. 자주 막히는 지점| 증상 | 원인 | 해결 ||---|---|---|| `CUDA out of memory` | VRAM 부족 | `TARGET`을 512로 낮추고, 4단계의 `enable_model_cpu_offload()`가 켜져 있는지 확인. 런타임 재시작 후 재실행 || 골격이 텅 빈 검은 이미지 | 인물 인식 실패 | 전신이 크게 보이는 사진으로 교체. `detect_resolution=768`로 올려보기 || `pipe.to()` 관련 오류 | 오프로딩과 수동 이동 충돌 | 4단계에서 `pipe.to("cuda")`를 따로 부르지 않는지 확인 || `ModuleNotFoundError: matplotlib` | 손 그리기에 필요 | `%pip install matplotlib` (Colab은 기본 설치) || 자세가 참조대로 안 나옴 | `pose_scale` 낮음 | 1.0 → 1.3 || 결과가 뻣뻣하고 배경이 단조로움 | `pose_scale` 높음 | 1.0 → 0.7 || 손가락이 이상함 | steps 부족 | `steps=35`, 네거티브에 `bad hands` 유지 || 얼굴이 뒤를 봄 | 얼굴 키포인트 없음 | 3단계 `include_face=True` 확인 || 사람이 두 명 나옴 | 참조 사진에 인물 2명 | 골격 이미지를 크롭하거나 1명만 있는 사진 사용 || 그래프 제목이 `□□□` | matplotlib에 한글 폰트 없음 | 제목·라벨을 영문으로 쓰기. 한글이 꼭 필요하면 `!apt-get install -y fonts-nanum` 후 런타임 재시작 |## 더 해볼 것- `stable-diffusion-v1-5/stable-diffusion-v1-5` 자리에 다른 SD1.5 파인튜닝 모델을 넣으면 화풍이 바뀝니다 (ControlNet은 그대로 호환).- `controlnet=[openpose_cn, depth_cn]`처럼 리스트를 주면 포즈와 깊이를 함께 제어할 수 있습니다.- `pose_map_1.save("my_pose.png")`로 골격을 저장해 두면 다음에 추출 단계를 건너뛸 수 있습니다.

---## 9단계. 무엇을 바꿔 보았고, 어떻게 달라졌는가세 가지 조건을 각각 따로 바꿔 보았습니다. **실행한 뒤 직접 관찰한 내용으로아래 대괄호를 채우세요.** 위 실험 1·2·3의 출력 그림을 보고 적으면 됩니다.### 실험 1. 같은 포즈 + 프롬프트만 바꿈 (photo / knight / astronaut / anime)- 자세: [ 네 장 모두 유지되었는가? 어긋난 것이 있었는가? ]- 달라진 것: [ 인물, 옷, 배경, 화풍 중 무엇이 얼마나 바뀌었는가 ]- 특이점: [ 예: 갑옷처럼 몸을 덮는 옷은 팔 위치가 뭉개지기 쉬웠다 ]### 실험 2. 같은 프롬프트 + `pose_scale`만 바꿈 (0.5 / 1.0 / 1.5)- `0.5`: [ 관찰 ]- `1.0`: [ 관찰 ]- `1.5`: [ 관찰 ]- 결론: [ 어느 값을 기본으로 쓸 것인가 ]### 실험 3. 같은 프롬프트 + 포즈 사진만 바꿈- 포즈 1 (`images1.jpg`, [어떤 자세]): [ 결과 ]- 포즈 2 (`images2.jpg`, [어떤 자세]): [ 결과 ]- 관찰: [ 자세가 복잡해질수록 어떻게 달라지는가 ]### 정리 — 어느 쪽을 바꿨을 때 결과가 크게 달라지는가- 프롬프트를 바꾸면: [ 관찰 ]- 포즈를 바꾸면: [ 관찰 ]- 잘 따라오지 않는 부분: [ 관찰 — 손가락, 겹친 다리, 얼굴 방향 등 ]